# Selección pairwise + compensación (backlog #1, idea de HOPE) — Colab L4

**Pregunta:** ¿elegir las neuronas mirando **pares** (quién se repite con quién) le gana a elegirlas **una por una**,
cuando después igual compensamos?

- Hoy (`energy+recon`): cada neurona tiene su puntaje solo; se quedan las de mayor energía y luego el ridge reajusta.
- HOPE (arXiv 2609.18916) elige con términos de pares, pero **nunca compensa**.
- **`obs+recon`** hace las dos: elimina una neurona a la vez, la que menos daño hace **después** de reajustar a las
  demás (Optimal Brain Surgeon por capa, usando toda la matriz Gram). Si dos neuronas siempre se prenden juntas,
  tira una y la otra hace su trabajo.

Qwen3.5-0.8B, índice de **matemáticas** (60k tokens de GSM8K train), presupuestos 85 / 70 / 50 / 30 % de las neuronas
del MLP. Perplejidad (más bajo = mejor) en soluciones de GSM8K test y en WikiText.
**Meta a batir:** `energy+recon` a 30 % = math **7.18** (corrida anterior; el modelo completo da 5.33).

`obs` sin `+recon` = selección pairwise sola (lo que haría HOPE).

~20–30 min en L4. Resultado: `MyDrive/wsi_runs/FINALES/hope_pairwise_qwen35-08b.json`. Al final se apaga solo.

In [ ]:
MODEL = "Qwen/Qwen3.5-0.8B"
BUDGETS = "0.85,0.7,0.5,0.3"
CRITERIA = "mean,energy,obs"     # taylor ya se midió (empata con energy) y cuesta un pase con backward
RUN_NAME = "hope_pairwise_qwen35-08b"
REPO = "/content/wsi"
OUT = f"/content/{RUN_NAME}.json"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U transformers huggingface_hub
!pip install -q flash-linear-attention || echo "fla opcional: sigue sin ella (más lento)"
!python -c "import transformers; print('transformers', transformers.__version__)"

In [ ]:
import os, json, subprocess, datetime
from google.colab import drive
drive.mount("/content/drive")
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
os.makedirs(FINALES, exist_ok=True)
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git pull -q
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
assert "def obs_select" in open(f"{REPO}/scripts/experiment_index_criteria.py").read(), "el repo no tiene obs_select: falta push"
print("commit", COMMIT)

In [ ]:
!cd {REPO} && python -u scripts/experiment_index_criteria.py --model-dir {MODEL} --data-dir /content/data \
    --budgets {BUDGETS} --criteria {CRITERIA} --out {OUT}

In [ ]:
r = json.load(open(OUT))
f = r["full"]
print(f"Modelo completo: ppl math {f['math']:.2f} | ppl wiki {f['wiki']:.2f}   (obs tardó {r['obs_select_s']} s)\n")
for b, row in r["budgets"].items():
    print(f"=== {float(b):.0%} de las neuronas ===")
    for c, v in sorted(((c, v) for c, v in row.items() if c != "overlap_with_mean"), key=lambda cv: cv[1]["math"]):
        print(f"  {c:14s} math {v['math']:9.3f}   wiki {v['wiki']:9.3f}")
    e, o = row["energy+recon"]["math"], row["obs+recon"]["math"]
    print(f"  obs+recon vs energy+recon (math): {o - e:+.3f}  ->", "GANA obs" if o < e else "no gana")
    print("  solapamiento con 'mean':", row["overlap_with_mean"], "\n")
r["_final"] = {"complete": True, "model": MODEL, "commit": COMMIT, "notebook": "hope_pairwise_colab.ipynb",
               "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
               "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w") as fh:
    json.dump(r, fh, indent=1)
os.replace(dst + ".tmp", dst)
print("Guardado final en Drive:", dst)

In [ ]:
# Apagar el runtime: solo si el resultado final ya está en Drive
AUTO_APAGAR = True
import time
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    print("Resultado en FINALES. Apagando el runtime en 60 s...", flush=True)
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")